# Amazon Bedrock AgentCore Gateway 도구 응답의 민감한 데이터 마스킹

## 개요

이 노트북에서는 **Amazon Bedrock Guardrails**와 통합된 **Amazon Bedrock AgentCore Gateway 인터셉터**를 사용하여 도구 응답의 **개인 식별 정보(PII)를 자동으로 익명화**하는 방법을 살펴봅니다. 인터셉터는 도구 응답을 실시간으로 검사하고, 결과를 클라이언트에 반환하기 전에 Bedrock의 기본 PII 탐지 및 익명화 기능으로 민감한 데이터를 익명화하여 데이터 개인정보 보호 규정을 준수하도록 지원합니다.

### Gateway에서 민감한 데이터를 마스킹하는 이유

고객 데이터에 접근하는 AI 애플리케이션을 구축할 때는 민감한 정보를 보호해야 합니다.

- **규정 준수**: GDPR, HIPAA, PCI-DSS 및 기타 규제 요구 사항 충족
- **데이터 최소화**: 클라이언트에는 필요한 정보만 공개
- **중앙 집중식 보호**: 모든 도구에 익명화 규칙을 일관되게 적용
- **Zero Trust**: 민감한 데이터 보호를 다운스트림 시스템에 의존하지 않음
- **AI 기반 탐지**: 31개 이상의 PII 유형을 지원하는 Bedrock Guardrails의 고급 PII 탐지 활용

Gateway 인터셉터는 개별 도구 구현을 수정하지 않고도 어떤 도구가 반환한 데이터인지와 관계없이 PII를 익명화하는 **중앙 집중식 정책 적용 지점**을 제공합니다.

---

## 이 튜토리얼에서 다루는 내용

이 튜토리얼에서는 **Amazon Bedrock Guardrails**와 **RESPONSE 인터셉터**를 사용하여 PII 익명화를 구현합니다.

🔒 **PII 익명화(RESPONSE 인터셉터 + Bedrock Guardrails)**  
   - 포괄적인 탐지를 위해 31개 이상의 PII 유형으로 구성된 Bedrock Guardrail 생성
   - Gateway의 도구 응답 가로채기
   - Bedrock Guardrails를 적용하여 민감한 데이터(이메일, 전화번호, SSN, 신용카드, 주소 등) 탐지 및 익명화
   - 탐지된 PII를 익명화된 자리 표시자(예: `[EMAIL]`, `[PHONE]`)로 대체
   - 정제된 응답을 클라이언트에 반환

![PII 마스킹 아키텍처](images/PII-mask.png)

---

## Gateway 인터셉터를 사용하는 이유

Gateway 인터셉터를 사용하면 다음을 수행할 수 있습니다.

- **데이터 보호**: AI 기반 탐지를 사용하여 응답의 민감한 정보를 자동으로 익명화
- **규정 준수 적용**: 모든 도구에 일관된 데이터 보호 정책 적용
- **포괄적인 범위**: 이름, 주소, 금융 데이터, 건강 정보 등을 포함한 31개 이상의 PII 유형 탐지
- **감사 및 거버넌스**: 데이터 접근 및 익명화 이벤트 기록
- **응답 변환**: 도구를 변경하지 않고 전송 중인 데이터 수정
- **관리형 서비스**: 지속적으로 업데이트되는 Bedrock Guardrails의 PII 탐지 모델 활용

인터셉터는 **Gateway 계층**에 연결되므로 애플리케이션 코드를 수정하지 않고도 기반에 있는 **모든** MCP 서버 또는 도구에서 데이터를 보호합니다.

---

## 튜토리얼 세부 정보

| 정보                     | 세부 정보                                                                    |
|--------------------------|------------------------------------------------------------------------------|
| **튜토리얼 유형**        | 실습형                                                                       |
| **AgentCore 구성 요소**  | Amazon Bedrock AgentCore Gateway, Gateway 인터셉터, Bedrock Guardrails      |
| **Gateway 대상 유형**    | MCP Server(Lambda 기반 도구)                                                |
| **인터셉터 유형**        | AWS Lambda(RESPONSE)                                                        |
| **인바운드 인증 IdP**    | Amazon Cognito(CUSTOM_JWT authorizer)                                       |
| **데이터 보호**          | Amazon Bedrock Guardrails를 사용한 PII 익명화(31개 이상의 PII 유형)         |
| **튜토리얼 구성 요소**   | Gateway, Lambda 인터셉터, Bedrock Guardrails, Amazon Cognito, MCP 도구      |
| **튜토리얼 적용 분야**   | 여러 산업 분야(PII를 다루는 모든 산업에 적용 가능)                         |
| **예제 난이도**          | 중급                                                                         |
| **사용 SDK**             | boto3                                                                        |

---

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.

- Jupyter Notebook(Python kernel)
- 다음 권한이 있는 AWS 자격 증명:
  - AWS Lambda
  - AWS IAM
  - Amazon Cognito
  - Amazon Bedrock AgentCore 서비스(제어 영역)
  - Amazon Bedrock Guardrails (bedrock:CreateGuardrail, bedrock:ApplyGuardrail)
- Python 3.9 이상
- AWS Lambda, IAM 역할, Amazon Cognito, Amazon Bedrock Guardrails 및 Amazon Bedrock AgentCore Gateway에 대한 기본적인 이해

> ⚠️ **참고:** 마지막의 정리 섹션에서는 이 튜토리얼에서 생성한 AWS 리소스(Gateway, Lambda, IAM 역할, Guardrail 등)를 삭제합니다. 모든 리소스를 삭제할 준비가 되었을 때만 실행하세요.


---

## 파트 1: 설정 및 배포

### 1.0단계: 필수 종속성 설치

이 튜토리얼에 필요한 모든 Python 패키지를 설치합니다.

In [ ]:
!pip install -r requirements.txt

### 1.1단계: 필수 라이브러리 가져오기

In [ ]:
import boto3
import json
import time
import sys
from pathlib import Path
from datetime import datetime
from botocore.exceptions import ClientError

# utils를 사용할 수 있도록 상위 디렉터리를 경로에 추가
utils_dir = Path.cwd().parent
sys.path.insert(0, str(utils_dir))

import utils

print("✓ Libraries imported")

# 이 배포의 고유 식별자 생성
DEPLOYMENT_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
print(f"\nDeployment ID: {DEPLOYMENT_ID}")

### 1.2단계: 배포 변수 구성

In [ ]:
# 구성
REGION = "us-east-1"
LAMBDA_FUNCTION_NAME = f"interceptor-lambda-{DEPLOYMENT_ID}"
LAMBDA_ROLE_NAME = f"interceptor-lambda-role-{DEPLOYMENT_ID}"
GATEWAY_NAME = f"interceptor-gateway-{DEPLOYMENT_ID}"

# 클라이언트 초기화
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
cognito_client = boto3.client("cognito-idp", region_name=REGION)

print("Configuration:")
print(f"  Lambda Function: {LAMBDA_FUNCTION_NAME}")
print(f"  Lambda Role: {LAMBDA_ROLE_NAME}")
print(f"  Gateway Name: {GATEWAY_NAME}")
print(f"  Region: {REGION}")

### 1.3단계: 민감한 데이터 필터링을 위한 Bedrock Guardrails 설정

Amazon Bedrock의 기본 제공 기능을 사용하여 PII를 익명화하도록 민감한 데이터 필터가 적용된 Bedrock Guardrail을 생성합니다.

In [ ]:
# 민감한 데이터 필터링을 위한 Bedrock Guardrails 설정
print("Creating Bedrock Guardrail with sensitive data filters...")

bedrock_client = boto3.client("bedrock", region_name=REGION)

GUARDRAIL_NAME = f"pii-masking-guardrail-{DEPLOYMENT_ID}"

# ANONYMIZE 동작을 사용하는 민감한 데이터 필터 정의
# Bedrock Guardrails는 기본적으로 31개의 PII 엔터티 유형을 지원
# ANONYMIZE 작업은 PII를 자리 표시자 텍스트로 대체(마스킹 동작)
sensitive_information_policy_config = {
    "piiEntitiesConfig": [
        # 일반 PII
        {"type": "ADDRESS", "action": "ANONYMIZE"},
        {"type": "AGE", "action": "ANONYMIZE"},
        {"type": "EMAIL", "action": "ANONYMIZE"},
        {"type": "NAME", "action": "ANONYMIZE"},
        {"type": "PHONE", "action": "ANONYMIZE"},
        {"type": "USERNAME", "action": "ANONYMIZE"},
        {"type": "PASSWORD", "action": "ANONYMIZE"},
        # 금융 정보
        {"type": "CREDIT_DEBIT_CARD_CVV", "action": "ANONYMIZE"},
        {"type": "CREDIT_DEBIT_CARD_EXPIRY", "action": "ANONYMIZE"},
        {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "ANONYMIZE"},
        {"type": "PIN", "action": "ANONYMIZE"},
        {"type": "INTERNATIONAL_BANK_ACCOUNT_NUMBER", "action": "ANONYMIZE"},
        {"type": "SWIFT_CODE", "action": "ANONYMIZE"},
        # 미국 전용 식별자
        {"type": "US_BANK_ACCOUNT_NUMBER", "action": "ANONYMIZE"},
        {"type": "US_BANK_ROUTING_NUMBER", "action": "ANONYMIZE"},
        {"type": "US_INDIVIDUAL_TAX_IDENTIFICATION_NUMBER", "action": "ANONYMIZE"},
        {"type": "US_PASSPORT_NUMBER", "action": "ANONYMIZE"},
        {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "ANONYMIZE"},
        {"type": "DRIVER_ID", "action": "ANONYMIZE"},
        # 영국 전용 식별자
        {"type": "UK_NATIONAL_HEALTH_SERVICE_NUMBER", "action": "ANONYMIZE"},
        {"type": "UK_NATIONAL_INSURANCE_NUMBER", "action": "ANONYMIZE"},
        {"type": "UK_UNIQUE_TAXPAYER_REFERENCE_NUMBER", "action": "ANONYMIZE"},
        # 캐나다 전용 식별자
        {"type": "CA_HEALTH_NUMBER", "action": "ANONYMIZE"},
        {"type": "CA_SOCIAL_INSURANCE_NUMBER", "action": "ANONYMIZE"},
        # 네트워크 및 기술 정보
        {"type": "IP_ADDRESS", "action": "ANONYMIZE"},
        {"type": "MAC_ADDRESS", "action": "ANONYMIZE"},
        {"type": "URL", "action": "ANONYMIZE"},
        # AWS 자격 증명
        {"type": "AWS_ACCESS_KEY", "action": "ANONYMIZE"},
        {"type": "AWS_SECRET_KEY", "action": "ANONYMIZE"},
        # 차량 식별 정보
        {"type": "VEHICLE_IDENTIFICATION_NUMBER", "action": "ANONYMIZE"},
        {"type": "LICENSE_PLATE", "action": "ANONYMIZE"},
    ]
}

try:
    # Guardrail 생성
    guardrail_response = bedrock_client.create_guardrail(
        name=GUARDRAIL_NAME,
        description="Guardrail for anonymizing sensitive PII data in tool responses",
        sensitiveInformationPolicyConfig=sensitive_information_policy_config,
        blockedInputMessaging="Input contains sensitive information that has been anonymized.",
        blockedOutputsMessaging="Output contains sensitive information that has been anonymized.",
    )

    GUARDRAIL_ID = guardrail_response["guardrailId"]
    GUARDRAIL_ARN = guardrail_response["guardrailArn"]

    print(f"✓ Guardrail created: {GUARDRAIL_ID}")
    print(f"  ARN: {GUARDRAIL_ARN}")

    # Guardrail 버전 생성
    print("\nCreating guardrail version...")
    version_response = bedrock_client.create_guardrail_version(
        guardrailIdentifier=GUARDRAIL_ID,
        description="Initial version with PII anonymization",
    )

    GUARDRAIL_VERSION = version_response["version"]
    print(f"✓ Guardrail version created: {GUARDRAIL_VERSION}")

    # 구성된 PII 유형 표시
    print(
        f"\n✓ Configured {len(sensitive_information_policy_config['piiEntitiesConfig'])} PII types for anonymization:"
    )
    for pii_config in sensitive_information_policy_config["piiEntitiesConfig"]:
        print(f"  - {pii_config['type']}: {pii_config['action']}")

except ClientError as e:
    error_code = e.response["Error"]["Code"]
    if error_code == "ConflictException":
        print(f"⚠ Guardrail with name '{GUARDRAIL_NAME}' already exists")
        # ID를 가져오기 위해 기존 Guardrail 목록 조회
        list_response = bedrock_client.list_guardrails()
        for guardrail in list_response.get("guardrails", []):
            if guardrail["name"] == GUARDRAIL_NAME:
                GUARDRAIL_ID = guardrail["id"]
                GUARDRAIL_ARN = guardrail["arn"]
                print(f"  Using existing guardrail: {GUARDRAIL_ID}")

                # 기존 Guardrail의 최신 버전 가져오기
                try:
                    get_response = bedrock_client.get_guardrail(guardrailIdentifier=GUARDRAIL_ID)
                    GUARDRAIL_VERSION = get_response.get("version", "DRAFT")
                    print(f"  Guardrail version: {GUARDRAIL_VERSION}")
                except Exception as get_error:
                    print(f"  ⚠ Could not get guardrail version: {get_error}")
                    GUARDRAIL_VERSION = "DRAFT"
                break
    else:
        print(f"✗ Failed to create guardrail: {e}")
        raise
except Exception as e:
    print(f"✗ Unexpected error creating guardrail: {e}")
    raise

### 1.4단계: Lambda 인터셉터용 IAM 역할 생성

Lambda에 실행 및 CloudWatch 로그 작성 권한을 부여합니다.

In [ ]:
# utils를 사용하여 Lambda 인터셉터용 IAM 역할 생성
print("Creating IAM role for Lambda interceptor...")

LAMBDA_ROLE_ARN = utils.create_lambda_role(
    role_name=LAMBDA_ROLE_NAME,
    description="Role for AgentCore Lambda Interceptor for PII masking with Bedrock Guardrails",
)

print(f"  ARN: {LAMBDA_ROLE_ARN}")

# Lambda 역할에 Bedrock Guardrails 권한 추가
print("\nAdding Bedrock Guardrails permissions to Lambda role...")
iam_client = boto3.client("iam")

bedrock_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Action": ["bedrock:ApplyGuardrail"], "Resource": "*"}],
}

try:
    iam_client.put_role_policy(
        RoleName=LAMBDA_ROLE_NAME,
        PolicyName="BedrockGuardrailsPolicy",
        PolicyDocument=json.dumps(bedrock_policy),
    )
    print("✓ Bedrock Guardrails permissions added")
    print("  Policy: bedrock:ApplyGuardrail on all resources")
except Exception as e:
    print(f"⚠ Failed to add Bedrock permissions: {e}")

### 1.4a단계: Bedrock Guardrail이 준비될 때까지 대기

Bedrock Guardrail이 전파되어 완전히 사용 가능한 상태가 될 때까지 기다립니다.

In [ ]:
time.sleep(10)

### 1.5단계: Lambda 인터셉터 함수 배포

Lambda가 도구 응답을 가로채고 Bedrock Guardrails를 사용하여 PII를 마스킹합니다.

In [ ]:
# utils를 사용하여 Lambda 인터셉터 배포
print("Deploying Lambda interceptor...")

# Lambda용 환경 변수 준비
lambda_env_vars = {}
if "GUARDRAIL_ID" in globals():
    lambda_env_vars["GUARDRAIL_ID"] = GUARDRAIL_ID
    lambda_env_vars["GUARDRAIL_VERSION"] = GUARDRAIL_VERSION
    print(f"  Configuring Lambda with Guardrail: {GUARDRAIL_ID} (version: {GUARDRAIL_VERSION})")
else:
    print("  ⚠ WARNING: GUARDRAIL_ID not found. Lambda will skip PII masking.")
    print("  Make sure to run Step 1.2a to create the Guardrail first.")

LAMBDA_ARN = utils.deploy_lambda_function(
    function_name=LAMBDA_FUNCTION_NAME,
    role_arn=LAMBDA_ROLE_ARN,
    lambda_code_path="src/lambda/lambda_function.py",
    description="AgentCore Response Lambda Interceptor to mask sensitive data using Bedrock Guardrails",
    timeout=30,
    memory_size=256,
    environment_vars=lambda_env_vars if lambda_env_vars else None,
    region=REGION,
)

print(f"  ARN: {LAMBDA_ARN}")

### 1.5a단계: Gateway에 Lambda 호출 권한 부여

Gateway가 Lambda 인터셉터 함수를 호출할 수 있도록 권한을 추가합니다.

In [ ]:
# Gateway에 Lambda 인터셉터 호출 권한 부여
print("\nGranting Gateway permission to invoke Lambda...")

utils.grant_gateway_invoke_permission(function_name=LAMBDA_FUNCTION_NAME, region=REGION)

### 1.6단계: Amazon Cognito 사용자 풀 및 앱 클라이언트 생성

OAuth 클라이언트 자격 증명 흐름을 사용하여 Gateway 인증용 Cognito 사용자 풀을 생성합니다.

In [ ]:
# utils를 사용하여 Gateway 인증용 Cognito 사용자 풀 및 클라이언트 생성
print("Creating Cognito User Pool and Client...")

USER_POOL_NAME = f"gateway-pool-{DEPLOYMENT_ID}"
RESOURCE_SERVER_ID = "gateway"
RESOURCE_SERVER_NAME = "Gateway Resource Server"
SCOPES = [{"ScopeName": "tools", "ScopeDescription": "Access to gateway tools"}]

# 사용자 풀 생성 또는 가져오기
USER_POOL_ID = utils.get_or_create_user_pool(cognito_client, USER_POOL_NAME)
print(f"  Pool ID: {USER_POOL_ID}")

# 리소스 서버 생성 또는 가져오기
utils.get_or_create_resource_server(cognito_client, USER_POOL_ID, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)

# 리소스 서버가 전파될 때까지 대기
print("  Waiting for resource server to propagate...")
time.sleep(3)

# 클라이언트 자격 증명 흐름으로 M2M 클라이언트 생성
CLIENT_NAME = f"gateway-client-{DEPLOYMENT_ID}"
CLIENT_ID, CLIENT_SECRET = utils.get_or_create_m2m_client(
    cognito_client,
    USER_POOL_ID,
    CLIENT_NAME,
    RESOURCE_SERVER_ID,
    SCOPES=[f"{RESOURCE_SERVER_ID}/tools"],
)

print(f"✓ User Pool Client created: {CLIENT_NAME}")
print(f"  Client ID: {CLIENT_ID}")
print(f"  Client Secret: {CLIENT_SECRET[:20]}...")

# OAuth URL 구성
POOL_DOMAIN = USER_POOL_ID.replace("_", "").lower()
COGNITO_DOMAIN = f"https://{POOL_DOMAIN}.auth.{REGION}.amazoncognito.com"
DISCOVERY_URL = f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"
TOKEN_URL = f"{COGNITO_DOMAIN}/oauth2/token"

print("\n✓ OAuth Configuration:")
print(f"  Discovery URL: {DISCOVERY_URL}")
print(f"  Token URL: {TOKEN_URL}")
print(f"  Scope: {RESOURCE_SERVER_ID}/tools")

### 1.7단계: RESPONSE 인터셉터가 적용된 Gateway 생성

**RESPONSE 인터셉터를 사용하는 이유**  
인터셉터는 실행 후 도구 응답을 처리하므로 클라이언트에 데이터를 반환하기 전에 PII를 마스킹할 수 있습니다.

In [ ]:
# Gateway IAM 역할 생성
gateway_iam_role = utils.create_agentcore_gateway_role_with_region(GATEWAY_NAME, REGION)
GATEWAY_ROLE_ARN = gateway_iam_role["Role"]["Arn"]

print(f"✓ Gateway role created: {GATEWAY_ROLE_ARN}")

# 역할이 전파될 때까지 대기
time.sleep(10)

# Lambda 인터셉터가 적용된 Gateway 생성
print("\nCreating Gateway with RESPONSE interceptor...")

try:
    gateway_response = gateway_client.create_gateway(
        name=GATEWAY_NAME,
        protocolType="MCP",
        protocolConfiguration={"mcp": {"supportedVersions": ["2025-03-26"]}},
        interceptorConfigurations=[
            {
                "interceptor": {"lambda": {"arn": LAMBDA_ARN}},
                "interceptionPoints": ["RESPONSE"],
                "inputConfiguration": {"passRequestHeaders": True},
            }
        ],
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": DISCOVERY_URL,
                "allowedClients": [CLIENT_ID],
            }
        },
        roleArn=GATEWAY_ROLE_ARN,
    )

    GATEWAY_ID = gateway_response.get("gatewayId")
    print(f"✓ Gateway created: {GATEWAY_ID}")

except Exception as e:
    print(f"\n✗ Failed to create Gateway: {e}")
    raise

### 1.8단계: Gateway가 준비될 때까지 대기

In [ ]:
# 서명된 요청을 사용하여 Gateway가 준비될 때까지 대기
print("\nWaiting for Gateway to be ready...")

max_attempts = 30
for attempt in range(max_attempts):
    try:
        response = gateway_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
        status_code = response.get("ResponseMetadata", {}).get("HTTPStatusCode")

        if status_code == 200:
            # gateway_info = response.json()
            status = response.get("status", "UNKNOWN")

            print(f"  [{attempt + 1}/{max_attempts}] Status: {status}")

            if status == "READY":
                GATEWAY_URL = response.get("gatewayUrl")
                print("\n✓ Gateway is ready!")
                print(f"  URL: {GATEWAY_URL}")

                # 인터셉터 구성 표시
                if "interceptorConfigurations" in response:
                    interceptor_configs = response["interceptorConfigurations"]
                    print("\n  Interceptor Configuration:")
                    for idx, config in enumerate(interceptor_configs):
                        print(f"    [{idx}] Interception Points: {config.get('interceptionPoints', [])}")
                        print(
                            f"    [{idx}] Lambda ARN: {config.get('interceptor', {}).get('lambda', {}).get('arn', 'N/A')}"
                        )
                        print(
                            f"    [{idx}] Pass Headers: {config.get('inputConfiguration', {}).get('passRequestHeaders', False)}"
                        )
                break
            elif status == "FAILED":
                print("\n✗ Gateway creation failed")
                print(f"  Details: {response}")
                raise Exception("Gateway failed")
        else:
            print(f"  [{attempt + 1}/{max_attempts}] HTTP Error: {response.status_code}")
    except Exception as e:
        print(f"  [{attempt + 1}/{max_attempts}] Error: {e}")

    time.sleep(10)
else:
    print("\n⚠ Timeout waiting for Gateway")
    raise Exception("Gateway timeout")

### 1.9단계: Gateway에 샘플 도구 등록

샘플 도구 Lambda(직원 데이터)를 배포하고 Gateway 대상으로 등록합니다.

In [ ]:
# 도구 Lambda를 배포하고 Gateway 대상으로 등록
print("Deploying tool Lambda functions...")

# 도구 모듈 가져오기
sys.path.insert(0, str(Path.cwd()))
from src.tools import employee_data_tool

# utils를 사용하여 도구 Lambda용 IAM 역할 생성
TOOL_ROLE_ARN = utils.create_lambda_role(
    role_name=f"tool-lambda-role-{DEPLOYMENT_ID}",
    description="Role for tool Lambda functions",
)

# 도구 Lambda 함수 배포
tools_to_deploy = [
    ("employee_data_tool", employee_data_tool),
]

deployed_tools = []

for tool_name, tool_module in tools_to_deploy:
    print(f"  Deploying {tool_name}...")

    function_name = f"{tool_name.replace('_', '-')}-{DEPLOYMENT_ID}"
    tool_code_path = Path(tool_module.__file__)

    lambda_arn = utils.deploy_lambda_function(
        function_name=function_name,
        role_arn=TOOL_ROLE_ARN,
        lambda_code_path=str(tool_code_path),
        environment_vars={"TOOL_NAME": tool_name},
        description=f"{tool_name} function",
        region=REGION,
    )

    tool_definition = getattr(
        tool_module,
        "TOOL_DEFINITION",
        {"name": tool_name, "description": f"{tool_name} function"},
    )

    deployed_tools.append(
        {
            "tool_name": tool_name,
            "function_name": function_name,
            "lambda_arn": lambda_arn,
            "tool_definition": tool_definition,
        }
    )

print(f"✓ Deployed {len(deployed_tools)} tool Lambdas")

# 도구를 Gateway 대상으로 등록
print("\nRegistering tools as Gateway targets...")
created_targets = []

for tool in deployed_tools:
    print(f"  Registering {tool['tool_name']}...")

    try:
        response = gateway_client.create_gateway_target(
            gatewayIdentifier=GATEWAY_ID,
            name=f"{tool['tool_name'].replace('_', '-')}-target",
            targetConfiguration={
                "mcp": {
                    "lambda": {
                        "lambdaArn": tool["lambda_arn"],
                        "toolSchema": {"inlinePayload": [tool["tool_definition"]]},
                    }
                }
            },
            credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
        )

        target_id = response["targetId"]
        print(f"    ✓ Target created: {target_id}")

        # 대상이 READY 상태가 될 때까지 대기
        for attempt in range(18):
            status_response = gateway_client.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=target_id)
            status = status_response.get("status")

            if status == "READY":
                print("    ✓ Target is READY")
                created_targets.append(
                    {
                        "tool_name": tool["tool_name"],
                        "target_id": target_id,
                        "lambda_arn": tool["lambda_arn"],
                    }
                )
                break
            elif status == "FAILED":
                print("    ✗ Target FAILED")
                break

            time.sleep(10)

    except Exception as e:
        print(f"    ✗ Failed to create target: {e}")

# 요약
print(f"\n✓ Deployed {len(deployed_tools)} tool Lambdas")
print(f"✓ Created {len(created_targets)} gateway targets")

if len(created_targets) < len(deployed_tools):
    print("⚠ Warning: Not all targets were created successfully")

# 정리를 위해 저장
DEPLOYED_TOOL_FUNCTIONS = [t["function_name"] for t in deployed_tools]
CREATED_TARGET_IDS = [t["target_id"] for t in created_targets]

---

## 파트 2: 테스트

### 2.1단계: Bedrock Guardrails를 사용한 PII 익명화 테스트

직원 데이터 도구를 호출하고 응답에서 PII가 익명화되는지 확인합니다.

#### 예상 결과

직원 데이터 도구는 다양한 유형의 PII가 포함된 실제와 유사한 직원 정보를 반환합니다. Lambda 인터셉터는 다음을 수행합니다.

1. 도구 실행 후 **응답 가로채기**
2. **Bedrock Guardrails를 적용**하여 31개 이상의 엔터티 유형에서 PII 탐지
3. 탐지된 PII를 자리 표시자 토큰으로 대체하여 **익명화**
4. 클라이언트에 **정제된 응답 반환**

#### Bedrock Guardrails 익명화 형식

Bedrock Guardrails는 탐지된 PII를 다음 형식의 익명화된 자리 표시자로 대체합니다.

- **이메일**: `john.doe@example.com` → `[EMAIL]`
- **전화번호**: `+1-555-123-4567` → `[PHONE]`
- **이름**: `John Doe` → `[NAME]`
- **주소**: `123 Main St, Springfield, IL 62701` → `[ADDRESS]`
- **SSN**: `123-45-6789` → `[US_SOCIAL_SECURITY_NUMBER]`
- **신용카드**: `4532-1234-5678-9010` → `[CREDIT_DEBIT_CARD_NUMBER]`
- **IP 주소**: `192.168.1.1` → `[IP_ADDRESS]`
- **URL**: `https://example.com` → `[URL]`

#### 출력 예제

**익명화 전(원본 도구 응답):**
```json
{
  "employee_id": "EMP-98765",
  "department": "Engineering",
  "contact_info": "alice.smith@company.com",
  "mailing_info": "456 Oak Avenue, Boston, MA 02101",
  "status": "Active"
}
```

**익명화 후(가로챈 응답):**
```json
{
  "employee_id": "EMP-98765",
  "department": "Engineering",
  "contact_info": "[EMAIL]",
  "mailing_info": "[ADDRESS]",
  "status": "Active"
}
```

`employee_id`, `department`, `status`와 같은 민감하지 않은 데이터는 변경되지 않고, 모든 PII(이메일, 주소)는 익명화된 자리 표시자로 대체됩니다.



In [ ]:
# 도구를 호출하여 PII 마스킹 테스트
import requests

print("Testing PII masking interceptor...")
print(f"Gateway URL: {GATEWAY_URL}")

# OAuth 토큰 가져오기
token_data = utils.get_token(
    user_pool_id=USER_POOL_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string="gateway/tools",
    REGION=REGION,
)

if "error" in token_data:
    print(f"✗ Token request failed: {token_data['error']}")
else:
    token = token_data["access_token"]
    print("✓ Token obtained")

### 2.2단계: 직원 데이터 도구 테스트

직원 데이터 도구를 호출하여 필드 이름에 민감성이 명시적으로 드러나지 않더라도 Bedrock Guardrails가 연락처 정보와 금융 데이터를 비롯한 다양한 PII를 어떻게 익명화하는지 확인합니다.

직원 데이터 도구는 다음을 반환합니다.
- **연락처 정보**: 이메일 및 실제 주소
- **금융 정보**: 은행 계좌 번호, 라우팅 번호, 신용카드 세부 정보, CVV, 만료일, PIN 및 납세자 식별 번호
- **민감하지 않은 데이터**: 직원 ID, 부서, 상태, 계좌 잔액, 신용 점수

**익명화 전:**
```json
{
  "employee_id": "EMP-98765",
  "department": "Engineering",
  "contact_info": "alice.smith@company.com",
  "mailing_info": "456 Oak Avenue, Boston, MA 02101",
  "status": "Active",
  "financial_info": {
    "bank_account": "123456789",
    "routing_number": "987654321",
    "credit_card": "4532-1234-5678-9010",
    "cvv": "123",
    "card_expiry": "12/28",
    "pin": "1234",
    "tax_id": "987-65-4321",
    "account_balance": 25000.50,
    "credit_score": 750
  }
}
```

**익명화 후:**
```json
{
  "employee_id": "EMP-98765",
  "department": "Engineering",
  "contact_info": "[EMAIL]",
  "mailing_info": "[ADDRESS]",
  "status": "Active",
  "financial_info": {
    "bank_account": "[US_BANK_ACCOUNT_NUMBER]",
    "routing_number": "[US_BANK_ROUTING_NUMBER]",
    "credit_card": "[CREDIT_DEBIT_CARD_NUMBER]",
    "cvv": "[CREDIT_DEBIT_CARD_CVV]",
    "card_expiry": "[CREDIT_DEBIT_CARD_EXPIRY]",
    "pin": "[PIN]",
    "tax_id": "[US_INDIVIDUAL_TAX_IDENTIFICATION_NUMBER]",
    "account_balance": 25000.50,
    "credit_score": 750
  }
}
```

**주요 관찰 사항:**
- **콘텐츠 기반 탐지**: `contact_info` 및 `mailing_info`와 같은 필드 이름에는 "email" 또는 "address"가 명시되어 있지 않지만, Bedrock Guardrails는 패턴 인식을 기반으로 콘텐츠를 탐지하고 익명화합니다.
- **포괄적인 금융 PII 보호**: 모든 민감한 금융 데이터(은행 계좌, 신용카드, 납세자 식별 번호)를 자동으로 탐지하고 익명화합니다.
- **선택적 익명화**: 계좌 잔액 및 신용 점수와 같은 민감하지 않은 금융 데이터는 변경되지 않습니다.
- **31개 이상의 PII 유형**: Bedrock Guardrails는 명시적인 필드 이름 일치 없이도 광범위한 PII 유형을 보호합니다.

In [ ]:
# 직원 데이터 도구 테스트
print("\n" + "=" * 60)
print("Testing Employee Data Tool with PII Anonymization")
print("=" * 60)

# 이전 단계의 토큰 재사용
if "token" in locals():
    print("✓ Using existing token")

    # 직원 데이터 도구 호출
    mcp_request = {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "id": 2,
        "params": {
            "name": "employee-data-tool-target___employee_data_tool",
            "arguments": {"employee_id": "EMP-98765"},
        },
    }

    response = requests.post(
        GATEWAY_URL,
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        },
        json=mcp_request,
    )

    if response.status_code == 200:
        result = response.json()
        print("\n✓ Employee tool invoked successfully")
        print("\nResponse (PII in contact_info and mailing_info should be anonymized):")
        print(json.dumps(result, indent=2))

        # 익명화 결과 강조
        print("\n📝 Notice:")
        print("  - 'employee_id', 'department', and 'status' remain unchanged (non-sensitive)")
        print("  - 'contact_info' email is replaced with [EMAIL] placeholder")
        print("  - 'mailing_info' address is replaced with [ADDRESS] placeholder")
        print("  - All financial PII is anonymized:")
        print("    • Bank account → [US_BANK_ACCOUNT_NUMBER]")
        print("    • Routing number → [US_BANK_ROUTING_NUMBER]")
        print("    • Credit card → [CREDIT_DEBIT_CARD_NUMBER]")
        print("    • CVV → [CREDIT_DEBIT_CARD_CVV]")
        print("    • Card expiry → [CREDIT_DEBIT_CARD_EXPIRY]")
        print("    • PIN → [PIN]")
        print("    • Tax ID → [US_INDIVIDUAL_TAX_IDENTIFICATION_NUMBER]")
        print("  - Non-sensitive financial data preserved (balances, credit scores, currency)")
        print("\n  ⭐ Key Point: Bedrock Guardrails uses content-based detection, not field names.")
        print("  Field names like 'mailing_info' don't explicitly say 'address', but the content")
        print("  is still detected and anonymized. This works across 31+ PII types automatically!")
    else:
        print(f"✗ Request failed: {response.status_code}")
        print(f"Response: {response.text}")
else:
    print("✗ No token available. Please run Step 2.1 first.")

---

# 파트 3: 정리 - 모든 리소스 삭제

⚠️ **경고: 파트 1에서 생성한 모든 리소스가 삭제됩니다!**

모든 리소스를 정리하려는 경우에만 이 섹션을 실행하세요.

### 3.1단계: 생성된 리소스 삭제

In [ ]:
# 정리 - utils를 사용하여 생성된 모든 리소스 삭제
print("Starting cleanup...")

# 1. Gateway 대상 삭제
if "CREATED_TARGET_IDS" in globals() and "GATEWAY_ID" in globals():
    utils.delete_gateway_targets(gateway_client, GATEWAY_ID, CREATED_TARGET_IDS)
    # Gateway를 삭제하기 전에 대상 삭제가 완료될 때까지 대기
    time.sleep(5)

# 2. Gateway 삭제
if "GATEWAY_ID" in globals():
    utils.delete_gateway(gateway_client, GATEWAY_ID)
    print("✓ Deleted gateway")

# 3. Lambda 함수 삭제(도구 + 인터셉터)
lambda_functions_to_delete = []
if "DEPLOYED_TOOL_FUNCTIONS" in globals():
    lambda_functions_to_delete.extend(DEPLOYED_TOOL_FUNCTIONS)
if "LAMBDA_FUNCTION_NAME" in globals():
    lambda_functions_to_delete.append(LAMBDA_FUNCTION_NAME)

if lambda_functions_to_delete:
    utils.delete_lambda_functions(lambda_functions_to_delete, REGION)

# 4. IAM 역할 삭제
if "LAMBDA_ROLE_NAME" in globals():
    utils.delete_iam_role(LAMBDA_ROLE_NAME)
if "DEPLOYMENT_ID" in globals():
    utils.delete_iam_role(f"tool-lambda-role-{DEPLOYMENT_ID}")
    utils.delete_iam_role(f"agentcore-{GATEWAY_NAME}-role")

# 5. Cognito 사용자 풀 삭제
if "USER_POOL_ID" in globals():
    utils.delete_cognito_user_pool(USER_POOL_ID, REGION)

# 6. Bedrock Guardrail 삭제
if "GUARDRAIL_ID" in globals():
    try:
        print("\nDeleting Bedrock Guardrail...")
        bedrock_client.delete_guardrail(guardrailIdentifier=GUARDRAIL_ID)
        print(f"✓ Deleted guardrail: {GUARDRAIL_ID}")
    except Exception as e:
        print(f"⚠ Failed to delete guardrail: {e}")

print("\n✓ Cleanup complete!")

---

# 요약

이 노트북에서는 Lambda 인터셉터를 사용한 PII 마스킹을 살펴봤습니다.

1. ✅ **설정** - Lambda 인터셉터, IAM 역할, Cognito 및 Gateway 생성
2. ✅ **테스트** - Gateway 응답을 통한 PII 마스킹 확인
3. ✅ **정리** - 모든 리소스 삭제

## 살펴본 내용

- 도구 응답에서 민감한 데이터를 마스킹하는 **Lambda RESPONSE 인터셉터**
- 정규식 패턴을 사용한 **PII 탐지 및 마스킹**
- 사용자 지정 인터셉터를 사용한 **Gateway 통합**
- **전체 리소스 수명 주기** 관리

## 다음 단계

- 사용 사례에 맞게 마스킹 패턴 사용자 지정
- 더 정교한 PII 탐지 추가(예: AWS Comprehend)
- 규정 준수 로깅과 통합
- 디버깅을 위해 CloudWatch 로그 모니터링